### Ideally this notebook will be the testbed for a quick script comparing pre and post fit parfiles when including noise processes

I have changed PINT so, so much that this can not be guaranteed to work and so I have to do a bunch of testing

08/09 11:24 AM: I have just begun this, does not yet work

In [1]:
import numpy as np
import sys
import astropy.units as u  
import matplotlib.pyplot as plt 
import pandas as pd
import astropy.time as time 

import pint
from pint.models import *

import pint.fitter
from pint.residuals import Residuals
from pint.toa import get_TOAs
import pint.logging
import pint.config


In [2]:
psrname = "J1909-3744"

partim_noise = "/home/mattm/projects/MPTA/data_noise_parfile_check/partim_w_noise/"
partim_blank = "/home/mattm/projects/MPTA/data_noise_parfile_check/partim/"

parfile_blank = partim_blank + psrname + "_tdb.par"
timfile_blank = partim_blank + psrname + ".tim"

parfile_noise = partim_noise + psrname + "_tdb_noise.par"
timfile_noise = partim_noise + psrname + ".tim"



In [3]:
## Functions to make the code able to handle all the signals and eventually to do an ADS


def weighted_average(dataframe, value, weight):
    val = dataframe[value]
    wt = dataframe[weight]

    return np.average(val, weights = 1/(np.array(wt)**2))


def uncertainty_scaled(dataframe, value):
    val = dataframe[value]

    return np.sqrt(np.average(val**2, weights = 1/(np.array(val)**2))) /np.sqrt(len(val))


def ecorr_apply(dataframe,value, ecorr):
    val = dataframe[value]
    
    return np.sqrt(val**2 + ecorr**2)

def chrom_yearly_sinusoid(toas, freqs, log10_Amp, phase, idx):
    """
    Chromatic annual sinusoid.
    :param log10_Amp: amplitude of sinusoid
    :param phase: initial phase of sinusoid
    :param idx: index of chromatic dependence
    :return wf: delay time-series [s]
    """

    wf = 10**log10_Amp * np.sin(2 * np.pi * econst.fyr * toas + phase)
    return wf * (1400 / freqs) ** idx

def chrom_gaussian_bump(toas, freqs, log10_Amp=-2.5, sign_param=1.0,
                    t0=53890, sigma=81, idx=2):
    """
    Chromatic time-domain Gaussian delay term in TOAs.
    Example: J1603-7202 in Lentati et al, MNRAS 458, 2016.
    """
    #t0 *= const.day
    #sigma *= const.day
    wf = 10**log10_Amp * np.exp(-(toas - t0)**2/2/sigma**2)
    return np.sign(sign_param) * wf * (1400 / freqs) ** idx

In [ ]:
## Use blank as the comparison for everything else, noise for the fit to the noise

m_blank, t_blank = get_model_and_toas(parfile_blank, timfile_blank, allow_name_mixing=True, planets=True)


m_noise, t_noise = get_model_and_toas(parfile_noise, timfile_noise, allow_name_mixing=True, planets=True)
psrname = m_noise.name.split("/")[-1].replace("_tdb_noise.par","")

/home/mattm/soft/PINT/src/pint/models/model_builder.py:220: UserWarning: Unrecognized parfile line 'EPHVER 5'
  warnings.warn(f"Unrecognized parfile line '{p_line}'", UserWarning)
/home/mattm/soft/PINT/src/pint/models/model_builder.py:220: UserWarning: Unrecognized parfile line 'TNSUBTRACTPOLY 1'
  warnings.warn(f"Unrecognized parfile line '{p_line}'", UserWarning)
/home/mattm/soft/PINT/src/pint/models/model_builder.py:220: UserWarning: Unrecognized parfile line 'DM_SERIES TAYLOR'
  warnings.warn(f"Unrecognized parfile line '{p_line}'", UserWarning)
2025-09-08 11:50:16.654 | DEBUG    | pint.toa:get_TOAs:195 - Using EPHEM = DE440 from the given model
2025-09-08 11:50:16.655 | DEBUG    | pint.toa:get_TOAs:211 - Using CLOCK = BIPM2024 from the given model


2025-09-08 11:50:17.708 | DEBUG    | pint.toa:__init__:1377 - No pulse number flags found in the TOAs
2025-09-08 11:50:17.716 | DEBUG    | pint.toa:apply_clock_corrections:2232 - Applying clock corrections (include_bipm = True)
2025-09-08 11:50:17.911 | INFO     | pint.observatory:gps_correction:230 - Applying GPS to UTC clock correction (~few nanoseconds)
2025-09-08 11:50:17.912 | DEBUG    | pint.observatory:_load_gps_clock:108 - Loading global GPS clock file
2025-09-08 11:50:17.914 | DEBUG    | pint.observatory.clock_file:__init__:812 - Global clock file gps2utc.clk saving kwargs={'bogus_last_correction': False, 'valid_beyond_ends': False}
2025-09-08 11:50:17.916 | DEBUG    | pint.observatory.clock_file:read_tempo2_clock_file:463 - Loading TEMPO2-format observatory clock correction file gps2utc.clk (/home/mattm/.astropy/cache/download/url/d3c81b5766f4bfb84e65504c8a453085/contents) with bogus_last_correction=False
2025-09-08 11:50:17.925 | INFO     | pint.observatory:find_clock_file:9

In [ ]:
## Get rid of the deterministic noise first